# COP vs Exergy Efficiency Figures (New)

This notebook generates four figure variants of the COP–Exergy efficiency map, based on `dual_setpoint_figures_plot` data and design language.

## Initial Setup

In [22]:
# Import required libraries
import sys
sys.path.append('src')
import enex_analysis as enex
import matplotlib.pyplot as plt
import dartwork_mpl as dm
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.ticker import AutoMinorLocator
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


## Plot Style Configuration

In [23]:
# ===============================
# Plot Style Settings
# ===============================
dm.style.use('scientific')
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Roboto:italic:regular'
plt.rcParams['font.size'] = 8.5

fs = {
    'label': dm.fs(-1),
    'tick': dm.fs(-1.5),
    'legend': dm.fs(-2.5),
    'subtitle': dm.fs(0),
    'cbar_tick': dm.fs(-2.5),
    'cbar_label': dm.fs(-1),
    'cbar_title': dm.fs(-2),
    'setpoint': dm.fs(-2),
    'text': dm.fs(-3.0),
    'cell_text': dm.fs(-2.5),
}

pad = {'label': 6, 'tick': 3}
LW = np.arange(0.25, 3.0, 0.25)

print("Plot style configured successfully.")

Plot style configured successfully.


## Data Loading and Preprocessing

In [24]:
# Load the CSV file (dual setpoint: cooling 26°C, heating 20°C)
data_file = '../data/dual_setpoint_small_office_hour.csv'
weekday_df = pd.read_csv(data_file)

cols = list(weekday_df.columns)
date_col = None
heating_col = None
cooling_col = None

for c in cols:
    if c.strip() == 'Date/Time' or 'Date/Time' in c:
        date_col = c
        break
for c in cols:
    if 'DistrictHeatingWater:Facility' in c:
        heating_col = c
        break
for c in cols:
    if 'DistrictCooling:Facility' in c:
        cooling_col = c
        break

weekday_df['Date/Time_clean'] = weekday_df[date_col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
weekday_df['Relative_Day_Index'] = weekday_df.index // 24
weekday_df['Day'] = (6 + weekday_df['Relative_Day_Index']) % 7

if cooling_col is not None:
    weekday_df.loc[weekday_df['Day'].isin([5, 6]), cooling_col] = 0
if heating_col is not None:
    weekday_df.loc[weekday_df['Day'].isin([5, 6]), heating_col] = 0

weekday_df['Month'] = weekday_df['Date/Time_clean'].str.slice(0, 2).astype(int)
if cooling_col is not None:
    weekday_df.loc[weekday_df['Month'].isin([1, 2, 3, 10, 11, 12]), cooling_col] = 0
if heating_col is not None:
    weekday_df.loc[weekday_df['Month'].isin([4, 5, 6, 7, 8, 9]), heating_col] = 0

weekday_df.reset_index(drop=True, inplace=True)
if heating_col is not None:
    weekday_df[heating_col] = pd.to_numeric(weekday_df[heating_col], errors='coerce').fillna(0.0)
if cooling_col is not None:
    weekday_df[cooling_col] = pd.to_numeric(weekday_df[cooling_col], errors='coerce').fillna(0.0)

print(f"Data loaded successfully. Shape: {weekday_df.shape}")
df = weekday_df

Data loaded successfully. Shape: (8760, 19)


## COP and Exergy Efficiency Calculation (Common)

In [39]:
Toa_list = df['Environment:Site Outdoor Air Drybulb Temperature [C](Hourly)']
cooling_load_list = df[cooling_col] * enex.s2h
heating_load_list = df[heating_col] * enex.s2h
cooling_load_list = cooling_load_list.clip(upper=21000)
heating_load_list = heating_load_list.clip(upper=21000)

ASHP_cooling_exergy_effi = []
ASHP_heating_exergy_effi = []
ASHP_cooling_COP = []
ASHP_heating_COP = []

for Toa, cooling_load, heating_load in zip(Toa_list, cooling_load_list, heating_load_list):
    if cooling_load > 0:
        ASHP_cooling = enex.AirSourceHeatPump_cooling()
        ASHP_cooling.T0 = Toa
        ASHP_cooling.T_a_room = 24
        ASHP_cooling.Q_r_int = cooling_load
        ASHP_cooling.Q_r_max = 21000
        ASHP_cooling.system_update()
        if ASHP_cooling.X_eff < 0:
            ASHP_cooling_exergy_effi.append(None)
            ASHP_cooling_COP.append(None)
        else:
            ASHP_cooling_exergy_effi.append(ASHP_cooling.X_eff)
            ASHP_cooling_COP.append(ASHP_cooling.COP_sys)
    else:
        ASHP_cooling_exergy_effi.append(None)
        ASHP_cooling_COP.append(None)

    if heating_load > 0:
        ASHP_heating = enex.AirSourceHeatPump_heating()
        ASHP_heating.T0 = Toa
        ASHP_heating.T_a_room = 21
        ASHP_heating.Q_r_int = heating_load
        ASHP_heating.Q_r_max = 21000
        ASHP_heating.system_update()
        if ASHP_heating.X_eff < 0:
            ASHP_heating_exergy_effi.append(None)
            ASHP_heating_COP.append(None)
        else:
            ASHP_heating_exergy_effi.append(ASHP_heating.X_eff)
            ASHP_heating_COP.append(ASHP_heating.COP_sys)
    else:
        ASHP_heating_exergy_effi.append(None)
        ASHP_heating_COP.append(None)

ASHP_cooling_COP_filtered = [c for c in ASHP_cooling_COP if c is not None]
ASHP_heating_COP_filtered = [c for c in ASHP_heating_COP if c is not None]
ASHP_cooling_exergy_effi_filtered = [e for e in ASHP_cooling_exergy_effi if e is not None]
ASHP_heating_exergy_effi_filtered = [e for e in ASHP_heating_exergy_effi if e is not None]

Toa_cooling_filtered = [t for t, e in zip(Toa_list, ASHP_cooling_exergy_effi) if e is not None]
Toa_heating_filtered = [t for t, e in zip(Toa_list, ASHP_heating_exergy_effi) if e is not None]

COP_c = np.array(ASHP_cooling_COP_filtered)
COP_h = np.array(ASHP_heating_COP_filtered)
X_eff_c_pct = np.array(ASHP_cooling_exergy_effi_filtered) * 100.0
X_eff_h_pct = np.array(ASHP_heating_exergy_effi_filtered) * 100.0
Toa_c = np.array(Toa_cooling_filtered)
Toa_h = np.array(Toa_heating_filtered)
Month_list = df['Month'].values
Month_cooling = np.array([m for m, e in zip(Month_list, ASHP_cooling_exergy_effi) if e is not None])
Month_heating = np.array([m for m, e in zip(Month_list, ASHP_heating_exergy_effi) if e is not None])

print(f"Cooling points: {len(COP_c)}, Heating points: {len(COP_h)}")

Cooling points: 1367, Heating points: 1519


## Figure 1: COP vs Exergy Efficiency Hexbin (1×2, Heating/Cooling)

In [94]:
import matplotlib as mpl

fig, axes = plt.subplots(1, 2, figsize=(dm.cm2in(17), dm.cm2in(7)))
plt.subplots_adjust(left=0.08, right=0.9, top=0.92, bottom=0.14, wspace=0.25)

COP_MIN, COP_MAX = 1.5, 6.0
X_EFF_MIN, X_EFF_MAX = 0, 35

# 0~1 사이 구간 중 시작과 끝값을 커스텀으로 지정해서 컬러맵 슬라이싱
vmin_norm = 0.2
vmax_norm = 1
ncolors = 256  # 더 부드러운 컬러맵을 위해 색상 세분화

color_slice = np.linspace(vmin_norm, vmax_norm, ncolors)

# dm.GnBu 없으면 GnBu 사용
base_cmap = plt.get_cmap("dm.GnBu")
colors = base_cmap(color_slice)
cmap_custom = mpl.colors.ListedColormap(colors)

max_per_bin_list = []

# continuous norm 적용
from matplotlib.colors import Normalize
norm_hex = Normalize(vmin=0, vmax=50)

for ax, COP, X_eff, title in zip(axes, [COP_c, COP_h], [X_eff_c_pct, X_eff_h_pct], ['(a) Cooling Mode', '(b) Heating Mode']):
    hb = ax.hexbin(
        COP,
        X_eff,
        gridsize=48,
        cmap=cmap_custom,
        mincnt=1,
        norm=norm_hex,
        linewidths=0.4,
        extent=(COP_MIN, COP_MAX, X_EFF_MIN, X_EFF_MAX),
        reduce_C_function=np.count_nonzero,
    )
    ax.set_xlabel('System COP (COP$_{sys}$)', fontsize=fs['label'], labelpad=pad['label'])
    ax.set_ylabel('Exergy Efficiency ($\\eta_{ex,sys}$) [%]', fontsize=fs['label'], labelpad=pad['label'])
    ax.set_xlim(COP_MIN, COP_MAX)
    ax.set_ylim(X_EFF_MIN, X_EFF_MAX)
    ax.tick_params(axis='both', which='major', labelsize=fs['tick'], pad=pad['tick'])
    ax.set_title(title, fontsize=fs['subtitle'])

    offsets = hb.get_offsets()
    counts = hb.get_array()
    if len(counts) > 0:
        max_count = counts.max()
        print(f"[{title}] bin별 최대 operating hours: {max_count}")
        max_per_bin_list.append(max_count)
    else:
        print(f"[{title}] 빈 hexbin입니다.")

fig.canvas.draw()
ax_pos = axes[0].get_position()
cbar_ax = fig.add_axes([0.92, ax_pos.ymin, 0.02, ax_pos.height])

cbar = fig.colorbar(
    hb, 
    cax=cbar_ax, 
    cmap=cmap_custom,
    norm=norm_hex,
)
cbar.set_label('Operating hours per bin', fontsize=fs['cbar_label'], labelpad=pad['label'])
cbar.ax.tick_params(labelsize=fs['cbar_tick'])
dm.save_and_show(fig)

[(a) Cooling Mode] bin별 최대 operating hours: 46.0
[(b) Heating Mode] bin별 최대 operating hours: 36.0


## Figure 2: Seasonal 2×2 COP vs Exergy (Scatter + KDE Contour)

In [ ]:
def add_kde_contours(ax, x, y, levels_pct=[0.2, 0.4, 0.6], color='k'):
    if len(x) < 10:
        return
    xy = np.vstack([x, y])
    kde = stats.gaussian_kde(xy)
    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()
    xx = np.linspace(xmin, xmax, 80)
    yy = np.linspace(ymin, ymax, 80)
    XX, YY = np.meshgrid(xx, yy)
    positions = np.vstack([XX.ravel(), YY.ravel()])
    Z = kde(positions).reshape(XX.shape)
    Z_flat = np.sort(Z.ravel())
    Z_flat = Z_flat[Z_flat > 0]
    if len(Z_flat) == 0:
        return
    level_vals = [np.percentile(Z_flat, (1 - p) * 100) for p in levels_pct]
    level_vals = sorted(level_vals)  # contour는 오름차순 levels 필요
    ax.contour(XX, YY, Z, levels=level_vals, colors=color, linewidths=0.8, alpha=0.8)

# 순서를 봄-여름-가을-겨울로 변경
seasons = [
    ('Spring (Mar–May)', [3, 4, 5], 'limegreen', 'Mixed modes'),
    ('Summer (Jun–Aug)', [6, 7, 8], 'crimson', 'Cooling'),
    ('Autumn (Sep–Nov)', [9, 10, 11], 'darkorange', 'Cooling'),
    ('Winter (Dec–Feb)', [12, 1, 2], 'teal', 'Heating cluster'),
]

fig, axes = plt.subplots(2, 2, figsize=(dm.cm2in(14), dm.cm2in(12)))
axes = axes.flatten()
plt.subplots_adjust(left=0.1, right=0.96, top=0.92, bottom=0.08, wspace=0.4, hspace=0.4)

labels = ['a)', 'b)', 'c)', 'd)']
for idx, (title, months, color, mode_label) in enumerate(seasons):
    ax = axes[idx]
    mc = np.isin(Month_cooling, months)
    mh = np.isin(Month_heating, months)
    COP_all = np.concatenate([COP_c[mc], COP_h[mh]])
    X_all = np.concatenate([X_eff_c_pct[mc], X_eff_h_pct[mh]])
    if len(COP_all) > 0:
        ax.scatter(COP_all, X_all, c=color, alpha=0.5, s=12, label=mode_label, edgecolors='none')
        add_kde_contours(ax, COP_all, X_all, color=color)
    ax.set_xlim(1.0, 6.5)
    ax.set_ylim(0, 35)
    ax.set_title(f"{labels[idx]} {title}", fontsize=fs['subtitle'])
    ax.set_xlabel('System COP (COP$_{sys}$)', fontsize=fs['label'], labelpad=pad['label'])
    ax.set_ylabel('Exergy Efficiency [%]', fontsize=fs['label'], labelpad=pad['label'])
    ax.tick_params(axis='both', labelsize=fs['tick'], pad=pad['tick'])
    ax.legend(fontsize=fs['legend'])

dm.save_and_show(fig)

## Figure 3: COP vs Exergy 1×2 Scatter (Environmental Temp. Color)

In [97]:
fig, axes = plt.subplots(1, 2, figsize=(dm.cm2in(17), dm.cm2in(7)))
plt.subplots_adjust(left=0.08, right=0.90, top=0.92, bottom=0.12, wspace=0.25)

for ax, COP, X_eff, Toa in zip(axes, [COP_c, COP_h], [X_eff_c_pct, X_eff_h_pct], [Toa_c, Toa_h]):
    sc = ax.scatter(COP, X_eff, c=Toa, cmap='coolwarm', alpha=1, s=15, vmin=-10, vmax=30)
    ax.set_xlabel('System COP (COP$_{sys}$)', fontsize=fs['label'], labelpad=pad['label'])
    ax.set_ylabel('Exergy Efficiency [%]', fontsize=fs['label'], labelpad=pad['label'])
    ax.set_xlim(1.0, 6.5)
    ax.set_ylim(0, 35)
    ax.tick_params(axis='both', labelsize=fs['tick'], pad=pad['tick'])

cbar_ax = fig.add_axes([0.92, 0.12, 0.02, 0.76])
cbar = fig.colorbar(sc, cax=cbar_ax)
cbar.set_label(r'Environmental Temp. (T$_0$) [°C]', fontsize=fs['cbar_label'], labelpad=pad['label'])
cbar.ax.tick_params(labelsize=fs['cbar_tick'])
dm.save_and_show(fig)

## Figure 4: Single Axes COP vs Exergy (Heating + Cooling)

In [ ]:
fig, ax = plt.subplots(figsize=(dm.cm2in(9), dm.cm2in(7)))
ax.scatter(COP_h, X_eff_h_pct, marker='^', c='teal', alpha=0.5, s=15, label='Heating', edgecolors='none')
ax.scatter(COP_c, X_eff_c_pct, marker='o', c='coral', alpha=0.5, s=15, label='Cooling', edgecolors='none')
ax.set_xlabel('System COP (COP$_{sys}$)', fontsize=fs['label'], labelpad=pad['label'])
ax.set_ylabel(r'Exergy Efficiency ($\eta_{ex,sys}$) [%]', fontsize=fs['label'], labelpad=pad['label'])
ax.set_xlim(1.5, 6.0)
ax.set_ylim(0, 35)
ax.tick_params(axis='both', labelsize=fs['tick'], pad=pad['tick'])
ax.legend(fontsize=fs['legend'])
ax.grid(True, alpha=0.3, linestyle='--')
dm.simple_layout(fig)
dm.save_and_show(fig)